![Ironhack logo](https://user-images.githubusercontent.com/23629340/40541063-a07a0a8a-601a-11e8-91b5-2f13e4e6b441.png)

# 02 | Processing and Database Design

*Project 1 — SQL: From Data to Insight*

**Team:** Zoé Doucet
**Dataset:** The Global Impacts Dataset of Invasive Alien Species (GIDIAS)

---
## 0. Setup

In [1]:
import sys
sys.path.append("..")          # so `src` is importable from inside notebooks/

import pandas as pd
import numpy as np


---
## 1. Load the raw data

The Global Impacts Dataset of Invasive Alien Species (GIDIAS) has first been published on the 21. of Mai 2025 by Springer Natur and authored by S. Bacher et al.  (https://doi.org/10.6084/m9.figshare.27908838)

In [2]:
df_raw = pd.read_csv("../data/raw/GIDIAS_20250417_machine_read.csv")

---
## 2. Clean

### 2.1 Discarding unecessary columns
Discarding all columns that contain free text, except `UniqueID`, as well as columns that caontain boolean values and are not necessary for the analysis.

Note: It would make more sense to standardise the column names first. However, since the column names are being compared with the names provided in the metadata, standardisation will be carried out later on.

In [3]:
df_meta = pd.read_excel("../data/raw/GIDIAS_metadata_20250507.xlsx")
free_text_columns = []
for column in df_raw:
    meta_row = df_meta.loc[df_meta["Variable"] == column] # finding right row the metadata for each column form the raw data   
    if meta_row["Number or range of unique entries"].iloc[0] == "Free text": # filtering the column for wich the number or range of unique entries is free text
        free_text_columns.append(column) # add the name of the column to the list of columns with free text
print(free_text_columns)
free_text_columns.remove("UniqueID") # remove "UniqueID" from the list of columns to remove


['UniqueID', 'IAS.Functional.Group', 'native.range.of.IAS', 'Text.excerpt', 'Methodology.details', 'Affected.native.species.Details', 'Affected.ecosystem.property', 'investigated.level.of.organization', 'details.NCP', 'magnitude.NCP', 'details.CWB', 'magnitude.CWB.details', 'IPLC']


In [4]:
# removing the columns with free text values 
df_clean = df_raw.drop(columns= free_text_columns)

In [5]:
# creating a list of columns with boolean type values to remove
boolean_columns = df_raw.select_dtypes(include="bool").columns
exclude = ["Island.k", "Protected.area.k", "global.extinction"] # these columns can be interesting for the analyses and will be kept
boolean_columns = boolean_columns.difference(exclude)
boolean_columns = boolean_columns.to_list()

In [6]:
df_clean = df_clean.drop(columns = boolean_columns)

### 2.2 Data quality corrections
Standardising the column names and correcting the data types

In [7]:
# using the column_names_std to standardise the column names 
from functions.cleaning import column_names_std
df_clean = column_names_std(df_clean) 
df_clean.nunique()

rowid                              22865
uniqueid                           22865
ias_species_name                    3779
verified_name_gbif_taxon            3353
gbif_scientificname_with_author     3204
genus                               2079
family                               761
order                                242
class                                 84
phylum                                36
kingdom                                8
ias_taxon                              4
reference                           6783
doi                                 1848
assessor                             106
year                                  87
year_of_impact                        92
type_of_source                        15
language                              16
region                                 5
country_location                     414
island                               368
island_k                               2
protected_area                       413
protected_area_k

In [8]:
# separating the suffix "id" from the word in front
df_clean.columns = df_clean.columns.str.replace(r"id$", "_id", regex=True) 

In [9]:
# correcting the data types to numeric
df_clean["year"] = pd.to_numeric(df_clean["year"].astype("Int64"))
df_clean["year_of_impact"] = pd.to_numeric(df_clean["year_of_impact"].astype("Int64"))
df_clean["magnitude_nature"] = pd.to_numeric(df_clean["magnitude_nature"].astype("Int64")) # converting first to numeric to drop the comma

In [10]:
# correcting the data type to string
df_clean["magnitude_nature"] = df_clean["magnitude_nature"].astype(str)

### 2.3. Discarding null values

As the dataset mostly contains categorical values, null values cannot be replaced by means or medians. For each case, it should be assessed whether the corresponding row should be discarded from the analysis. Rows containing null values in the columns `year`, `year_of_impact` and `region` may be excluded, as these values are essential for the analysis.

In [11]:
# making a list of all the rows with null values in the columns yaer, yaer of impact and region
nan_rows = df_clean[df_clean["region"].isna() & df_clean["year"].isna() & df_clean["year_of_impact"].isna()].index

In [12]:
df_clean = df_clean.drop(index = nan_rows)

Furthermore all row with null values in the column region will be discraded, as region will become a primary key.

In [13]:
nan_region = df_clean[df_clean["region"].isna()].index
df_clean = df_clean.drop(index = nan_region)

Saving the cleaned data ind data/clean as GIDIAS_cleaned

In [15]:
df_clean.to_csv("../data/clean/GIDIAS_cleaned.csv", index=False) # saving the cleaned data

---
## 3. Design your tables

The dataset will be split into three tables:

- an **impact** table with infomation regarding the impact a species has such as the magnitude of the impact, the realm it has an impact on and the language the impact was first described in
    - the primary key will be `unique_id`, which identifies the impact each species has in a distinct country and region
    - `species_id` will be one foreign key, wich will be created for each species
    - `location_id` will be an other foreign key, wich will be created for each region and country combination

- a **species** table with information regading the species such as family, class, kingdomm...
    - the primary key will be `species_id`

- the third table will be a table named **location**, which contains infomation on the location the species has an impact on, such as region and country
    - the primary key will be `location_id`


![Invasive Species ERD](../images/ERD_invasiv_species_1.png)

---
## 4. Building the tables

### 4.1 Creating the `year_probability` column

In [ ]:
df_clean["year_probability"] = df_clean[["year", "year_of_impact"]].min(axis=1)#adding the smallest number from "year" or "year_of_impact" to "year_probability"

### 4.2 Creating the foreign keys
Creating the column `species_id`, which will serve as a foreign key

In [45]:
df_clean["species_id"] = pd.factorize(df_clean["ias_species_name"])[0]

In [54]:
df_clean["species_id"].isna().sum() == 0

np.True_

`species_id` has no missing values.

Creating the column `location_id`, which will serve as a foreign key

In [ ]:
# creating a datframe with all possible combinations of region, country_location, island and islan_k
location_unique = df_clean[
    ["region", "country_location", "island", "island_k"]
].drop_duplicates().copy()

location_unique["location_id"] = range(1, len(location_unique) + 1)

In [122]:
# merging location_unique with df_clean
df_clean = df_clean.merge(
    location_unique,
    on=["region", "country_location", "island", "island_k"],
    how="left"
)

In [128]:
df_clean["location_id"].isna().sum() == 0

np.True_

`location_id` has no missing values.

### 4.3 Creating the tables

Creating the **species** table.

In [57]:
species = df_clean[[
"species_id",                        
"ias_species_name",   
"verified_name_gbif_taxon", 
"gbif_scientificname_with_author", 
"genus",                               
"family",                               
"order",                                
"class",                                 
"phylum",                                
"kingdom", 
"ias_taxon"]]

In [58]:
species.head()

,species_id,ias_species_name,verified_name_gbif_taxon,gbif_scientificname_with_author,genus,family,order,class,phylum,kingdom,ias_taxon
0,0,Gambusia affinis,Gambusia affinis,"Gambusia affinis (Baird & Girard, 1853)",Gambusia,Poeciliidae,Cyprinodontiformes,NaN,Chordata,Animalia,Vertebrate
1,1,Onchorhynchus mykiss,Oncorhynchus mykiss,"Oncorhynchus mykiss (Walbaum, 1792)",Oncorhynchus,Salmonidae,Salmoniformes,NaN,Chordata,Animalia,Vertebrate
2,2,Procambrus clarkii,Procambarus clarkii,"Procambarus clarkii (Girard, 1852)",Procambarus,Cambaridae,Decapoda,Malacostraca,Arthropoda,Animalia,Invertebrate
3,2,Procambrus clarkii,Procambarus clarkii,"Procambarus clarkii (Girard, 1852)",Procambarus,Cambaridae,Decapoda,Malacostraca,Arthropoda,Animalia,Invertebrate
4,3,Ctenopharyngodon idella,Ctenopharyngodon idella,"Ctenopharyngodon idella (Valenciennes, 1844)",Ctenopharyngodon,Cyprinidae,Cypriniformes,NaN,Chordata,Animalia,Vertebrate


In [108]:
# droping duplicates
species = species.drop_duplicates(subset = "species_id")

Creating the **impact** table.

In [151]:
impact = df_clean[[
"unique_id",
"species_id",
"location_id",
"reference",                        
"doi",                                 
"assessor",                             
"year",                                  
"year_of_impact",
"year_probability",                        
"type_of_source",   
"units_of_analysis_clean",               
"spatial_scale",                          
"affected_native_species_taxon",          
"mechanism_nature_clean",                
"direction_nature",                      
"direct_or_indirect_nature",              
"global_extinction",                      
"magnitude_nature",                       
"affected_ncp_clean",                    
"direction_ncp",                          
"affected_cwb_clean",                    
"direction_cwb",                          
"magnitude_cwb",      
"language",   
"protected_area",                       
"protected_area_k",                       
"realm"]]            

In [60]:
impact.head()

,unique_id,species_id,region,reference,doi,assessor,year,year_of_impact,type_of_source,units_of_analysis_clean,...,magnitude_nature,affected_ncp_clean,direction_ncp,affected_cwb_clean,direction_cwb,magnitude_cwb,language,protected_area,protected_area_k,realm
0,AF_Fish_crayfish_00001,0,Africa,Kruger_ D.J.D._ Hamer_ A.J. & Du Preez_ L.H. U...,10.1007/s11252-015-0443-y,Olaf L. F. Weyl,2015,2012,Peer-reviewed,Inland surface waters and water bodies/freshwater,...,1,NaN,NaN,NaN,NaN,NaN,English,NaN,False,Freshwater
1,AF_Fish_crayfish_00002,1,Africa,Shelton_ J.M._ Samways_ M.J. & Day_ J.A. Non-n...,10.1007/s10750-014-2067-2,Olaf L. F. Weyl,2015,<NA>,Peer-reviewed,Inland surface waters and water bodies/freshwater,...,2,NaN,NaN,NaN,NaN,NaN,English,NaN,False,Freshwater
2,AF_Fish_crayfish_00003,2,Africa,Harper_ D._ Smart_ A. C._ Coley_ S._ Schmitz_ ...,10.1023/A:1023330614984,Olaf L. F. Weyl,2002,<NA>,Peer-reviewed,Inland surface waters and water bodies/freshwater,...,2,NaN,NaN,NaN,NaN,NaN,English,NaN,False,Freshwater
3,AF_Fish_crayfish_00004,2,Africa,Hickley P. and Harper DM. 2001. Fish community...,10.1002/9780470995679.ch20.,Olaf L. F. Weyl,2001,<NA>,Peer-reviewed,Inland surface waters and water bodies/freshwater,...,2,NaN,NaN,NaN,NaN,NaN,English,NaN,False,Freshwater
4,AF_Fish_crayfish_00005,3,Africa,PSR Weyl & GD Martin (2016) Have grass carp dr...,10.2989/16085914.2015.1137856,Olaf L. F. Weyl,2016,<NA>,Peer-reviewed,Inland surface waters and water bodies/freshwater,...,2,NaN,NaN,NaN,NaN,NaN,English,NaN,False,Freshwater


Creating the **location** table.

In [130]:
location = df_clean[[
"location_id",
"region",
"country_location",
"island",
"island_k"]]

In [131]:
location.head()

,location_id,region,country_location,island,island_k
0,1,Africa,South Africa,NaN,False
1,1,Africa,South Africa,NaN,False
2,2,Africa,Kenya,NaN,False
3,2,Africa,Kenya,NaN,False
4,1,Africa,South Africa,NaN,False


In [132]:
# droping duplicates
location = location.drop_duplicates(subset = "location_id")


---
## 5. Check the foreign keys

**Do not skip this.** A dangling foreign key is the failure that costs people their Wednesday: the database loads without an error, then your joins silently drop rows and every number is quietly wrong. Check every key value exists in its parent table before you load anything.

Checking if every value from the foreign key `species_id` and `region` are present in the table impact.

In [133]:
missing_species = impact.loc[
    ~impact["species_id"].isin(species["species_id"]),
    "species_id"
].unique()
missing_species # every species_id in impact is in species

array([], dtype=int64)

In [136]:
missing_regions = impact.loc[
    ~impact["location_id"].isin(location["location_id"]),
    "location_id"
].unique()
missing_regions # every region in impact is in location

array([], dtype=int64)

---
## 6. Create the database and load it

Write your `CREATE TABLE` statements in `sql/schema.sql`, then load the tables — parents before children, or the foreign keys will be rejected.

### 6.1 Creating the database

In [77]:
import sqlite3
from pathlib import Path

In [162]:
project_con = sqlite3.connect("../project_inv_sp.db")

In [163]:
project_con.execute("PRAGMA foreign_keys = ON")

In [143]:
schema = Path("../sql/schema.sql").read_text()

In [144]:
project_con.executescript(schema)

In [145]:
project_con.commit()

In [159]:
project_con.close()

### 6.2 Loading the database

In [146]:
species.to_sql("species", project_con, if_exists ="append", index=False)

3718

In [148]:
location.to_sql("location", project_con, if_exists ="append", index=False)

899

In [154]:
impact.to_sql("impact", project_con, if_exists ="append", index=False)

22734

---
## 7. Verifying


### 7.1 Verifying the row counts

In [ ]:
# species
len(species) == project_con.execute("SELECT COUNT(*) FROM species;").fetchone()[0]

True

In [157]:
# locationes
len(location) == project_con.execute("SELECT COUNT(*) FROM location;").fetchone()[0]

True

In [158]:
# impact
len(impact) == project_con.execute("SELECT COUNT(*) FROM impact;").fetchone()[0]

True

The row counts matches for every table.

### 7.2 Verifying the join

In [165]:
check_join_i = pd.read_sql("SELECT unique_id FROM impact LIMIT 5;", project_con)
check_join_i

,unique_id
0,AF_Fish_crayfish_00001
1,AF_Fish_crayfish_00002
2,AF_Fish_crayfish_00003
3,AF_Fish_crayfish_00004
4,AF_Fish_crayfish_00005


In [166]:
check_join_s = pd.read_sql("SELECT s.genus FROM impact JOIN species AS s ON impact.species_id = s.species_id LIMIT 5;", project_con)
check_join_s

,genus
0,Gambusia
1,Oncorhynchus
2,Procambarus
3,Procambarus
4,Ctenopharyngodon


In [167]:
check_join_l = pd.read_sql("SELECT l.island FROM impact JOIN location AS l ON impact.location_id = l.location_id LIMIT 5;", project_con)
check_join_l

,island
0,None
1,None
2,None
3,None
4,None


The joins work.